# Instance-level visual search cascade

Experimental notebook: orchestration and result tables only. All logic lives in the `visual_search` package.

**Status:** step 2. Dataset loading, the evaluation harness and trivial baselines (random, color histogram). DINOv2 retrieval, masking, matching and the interactive UI come in later steps; see `docs/log/`.

In [ ]:
import logging
from pathlib import Path

import pandas as pd

from visual_search.cascade import Cascade
from visual_search.data import load_folder_dataset, make_synthetic_dataset, parse_dataset, split_dataset, validate_dataset
from visual_search.eval import comparison_table, evaluate
from visual_search.retrieval import ColorHistogramRetriever, RandomRetriever

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
SEED = 0

## Datasets
- **Synthetic** (SMOKE TEST ONLY): procedural textures in cluttered scenes. Useful to check that everything runs, not to compare methods.
- **Real**: `data/<instance_id>/*.jpg`, with distractors in `data/singletons/`. See the README.

In [ ]:
synthetic_split = split_dataset(
    make_synthetic_dataset(n_instances=8, views_per_instance=3, n_distractors=20, n_near_duplicates=2, seed=SEED),
    seed=SEED,
)
print("synthetic:", synthetic_split.summary())

real_records = load_folder_dataset(DATA_DIR) if DATA_DIR.is_dir() else []
print(validate_dataset(parse_dataset(real_records)).summary())
real_split = split_dataset(real_records, seed=SEED, strict=False) if real_records else None
print("real:", real_split.summary() if real_split else "no data")

## Baseline evaluation
Recall@k / MRR per stage and latency per stage (median / p95, ms). `random` is the chance level.

In [ ]:
def run_baselines(split):
    """Evaluate the trivial baselines on a split and return the reports."""
    return [evaluate(Cascade(RandomRetriever(seed=SEED)), split), evaluate(Cascade(ColorHistogramRetriever()), split)]

splits = {"synthetic": synthetic_split}
if real_split and real_split.queries:
    splits["real"] = real_split
reports = {name: run_baselines(split) for name, split in splits.items()}
pd.concat({name: comparison_table(reps) for name, reps in reports.items()}, names=["dataset"]).round(3)

## Per-query inspection (color histogram on synthetic)

In [ ]:
reports["synthetic"][1].per_query